<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Entropy_pooling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Entropy Pooling + Markowitz Portfolio Optimization
# NIFTY 50 Example
# ============================================================

# Install:
# pip install yfinance pandas numpy scipy cvxpy

import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import cvxpy as cp

# ------------------------------------------------------------
# 1. Download Data
# ------------------------------------------------------------

tickers = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "ITC.NS",
    "SBIN.NS",
    "LT.NS",
    "BHARTIARTL.NS",
    "HINDUNILVR.NS"
]

prices = yf.download(
    tickers,
    start="2022-01-01",
    end="2025-01-01",
    auto_adjust=True
)["Close"]

returns = prices.pct_change().dropna()

# ------------------------------------------------------------
# 2. Historical Scenarios
# ------------------------------------------------------------

R = returns.values

n_scenarios = len(R)

p = np.ones(n_scenarios) / n_scenarios

# ------------------------------------------------------------
# 3. Investor View
# RELIANCE expected return = 0.10% daily
# ------------------------------------------------------------

asset = 0

view_return = 0.001

# ------------------------------------------------------------
# 4. Entropy Pooling
# ------------------------------------------------------------

def entropy(q):
    q = np.clip(q,1e-12,None)
    return np.sum(q*np.log(q/p))

constraints = [

    {
        'type':'eq',
        'fun':lambda q: np.sum(q)-1
    },

    {
        'type':'eq',
        'fun':lambda q: np.dot(q,R[:,asset])-view_return
    }

]

bounds = [(1e-10,1)]*n_scenarios

result = minimize(
    entropy,
    p,
    method="SLSQP",
    constraints=constraints,
    bounds=bounds
)

q = result.x

# ------------------------------------------------------------
# 5. Updated Expected Returns
# ------------------------------------------------------------

entropy_mu = np.dot(q,R)

classical_mu = returns.mean().values

cov = returns.cov().values

# ------------------------------------------------------------
# 6. Classical Markowitz
# ------------------------------------------------------------

n_assets = len(tickers)

w1 = cp.Variable(n_assets)

problem1 = cp.Problem(

    cp.Maximize(

        classical_mu@w1
        -
        5*cp.quad_form(w1,cov)

    ),

    [

        cp.sum(w1)==1,

        w1>=0

    ]

)

problem1.solve()

weights_classical = w1.value

# ------------------------------------------------------------
# 7. Entropy Pooling Markowitz
# ------------------------------------------------------------

w2 = cp.Variable(n_assets)

problem2 = cp.Problem(

    cp.Maximize(

        entropy_mu@w2
        -
        5*cp.quad_form(w2,cov)

    ),

    [

        cp.sum(w2)==1,

        w2>=0

    ]

)

problem2.solve()

weights_entropy = w2.value

# ------------------------------------------------------------
# 8. Performance Function
# ------------------------------------------------------------

def performance(weights,mu,cov):

    ret = weights@mu

    risk = np.sqrt(weights.T@cov@weights)

    sharpe = ret/risk

    return ret,risk,sharpe

ret1,risk1,sharpe1 = performance(
    weights_classical,
    classical_mu,
    cov
)

ret2,risk2,sharpe2 = performance(
    weights_entropy,
    entropy_mu,
    cov
)

# ------------------------------------------------------------
# 9. Results
# ------------------------------------------------------------

comparison = pd.DataFrame({

    "Classical":weights_classical,

    "EntropyPooling":weights_entropy

},index=tickers)

print("\nPortfolio Weights")

print(comparison.round(4))

print("\n")

print("Classical Portfolio")

print("Expected Return :",round(ret1,5))

print("Risk            :",round(risk1,5))

print("Sharpe Ratio    :",round(sharpe1,4))

print("\n")

print("Entropy Pooling Portfolio")

print("Expected Return :",round(ret2,5))

print("Risk            :",round(risk2,5))

print("Sharpe Ratio    :",round(sharpe2,4))

[*********************100%***********************]  10 of 10 completed



Portfolio Weights
               Classical  EntropyPooling
RELIANCE.NS       0.3616          0.2733
TCS.NS            0.0000         -0.0000
INFY.NS          -0.0000         -0.0000
HDFCBANK.NS      -0.0000          0.0048
ICICIBANK.NS      0.0000         -0.0000
ITC.NS            0.5672          0.6283
SBIN.NS           0.0712          0.0936
LT.NS             0.0000         -0.0000
BHARTIARTL.NS    -0.0000          0.0000
HINDUNILVR.NS     0.0000         -0.0000


Classical Portfolio
Expected Return : 0.00126
Risk            : 0.00975
Sharpe Ratio    : 0.1294


Entropy Pooling Portfolio
Expected Return : 0.00115
Risk            : 0.00981
Sharpe Ratio    : 0.1177
